# Classify Job Postings

In the previous notebook, you scraped recent job postings and saved them to `jobs/1-scraped_jobs.jsonl`.

In this notebook, you will use an LLM to decide whether each posting is truly about AI engineering.

In [ ]:
import json
from dotenv import load_dotenv
from pathlib import Path

import pandas as pd
from openai import OpenAI
from IPython.display import HTML, display

In [ ]:
load_dotenv(override=True)

## Load the Scraped Jobs

This cell reads the JSONL file created by `3-scrape-job-postings.ipynb`.

If the file is missing, run the scraping notebook first. The repository also includes example data in the `jobs` folder, so you can continue even if live scraping failed on your machine.

In [ ]:
scraped_jobs_path = Path("jobs") / "1-scraped_jobs.jsonl"

if not scraped_jobs_path.exists():
    raise FileNotFoundError(
        f"Scraped jobs JSONL file not found: {scraped_jobs_path.resolve()}. Run 3-scrape-job-postings.ipynb first."
    )

if scraped_jobs_path.stat().st_size == 0:
    raise ValueError(
        "The scraped jobs JSONL file is empty. Run 3-scrape-job-postings.ipynb first."
    )

jobs_df = pd.read_json(scraped_jobs_path, lines=True)

print(f"Scraped jobs JSONL file loaded successfully with {len(jobs_df)} entries!")

## Create the Model Client

By default, this notebook uses the OpenAI API.

If you want to try the same workflow with a local model, uncomment the Ollama setup in the next cell and use a model name that you already pulled with `ollama pull`.

In [ ]:
client = OpenAI()

# Alternatively, if you want to use a local model 👇

# OLLAMA_BASE_URL = "http://localhost:11434/v1"

# client = OpenAI(
#    base_url=OLLAMA_BASE_URL,
#    api_key="ollama",
# )

## Define the Classification Instructions

The instructions tell the model what counts as an AI engineering role for this course.

In [ ]:
instructions = """
You classify whether a job posting is truly for an AI Engineering role.

AI Engineering definition:
- AI engineering means building applications on top of foundation models or in other words integrating them into products.
- Traditional ML engineering focuses on building, training, or tuning models; AI engineering primarily leverages existing models.
- MLOps and platform engineering are not AI engineering, as they focus on infrastructure and tooling rather than building AI-powered features.

Decision rules:
- Set is_ai_engineering_role to true when the main responsibility is building product or application features on top of foundation models or LLMs.
- Set is_ai_engineering_role to false when the role is mainly traditional software engineering, data science, analytics, ML research, model training, classical ML engineering, MLOps or platform work, or something else where AI application work is not the core responsibility.
- If the posting is ambiguous or unclear, set is_ai_engineering_role to false.
- In reason, briefly explain the main evidence for the decision in one sentence.
""".strip()

## Define the Output Schema

We ask the model for structured output: one boolean decision and one short reason.

That makes the response easy to parse, store in a DataFrame, and reuse in the next notebooks.

Reference: [json-schema.org](https://json-schema.org/)

In [ ]:
schema = {
    "type": "object",
    "properties": {
        "is_ai_engineering_role": {"type": "boolean"},
        "reason": {"type": "string"},
    },
    "required": ["is_ai_engineering_role", "reason"],
    "additionalProperties": False,
}

## Classify Each Posting

This loop sends one LLM request per job posting.

The title and description become the input data. The instructions and schema stay the same for every request, so each posting is judged against the same definition.

In [ ]:
results = []

for i, (_, job) in enumerate(jobs_df.iterrows(), start=1):
    title = job["title"]
    description = job["description"]

    print(f"Screening job {i}/{len(jobs_df)}: {title}")

    response = client.responses.create(
        model="gpt-5.4-mini",  # or local model like "gemma4:e4b"
        instructions=instructions,
        input=f"""Classify this job posting.\n\nTitle: {title}\n\nDescription:\n{description}""",
        text={
            "format": {
                "type": "json_schema",
                "name": "ai_engineering_job_screening",
                "schema": schema,
                "strict": True,
            },
            "verbosity": "low",
        },
    )

    classification = json.loads(response.output_text)

    results.append({
        "is_ai_engineering_role": classification["is_ai_engineering_role"],
        "llm_reason": classification["reason"],
    })

## Check the Classification Summary

Before saving the data, check how many postings the model accepted and rejected.

In [ ]:
results_df = pd.DataFrame(results)

ai_engineer_count = int(results_df["is_ai_engineering_role"].sum())
non_ai_engineer_count = int((~results_df["is_ai_engineering_role"]).sum())

print(f"Jobs screened by LLM: {len(results_df)}")
print(f"Jobs classified as AI engineering roles: {ai_engineer_count}")
print(f"Jobs classified as not AI engineering or unclear: {non_ai_engineer_count}")

## Combine Jobs and Classifications

Now we attach the model's decision back to the original scraped jobs.

This keeps the title, URL, and description next to `is_ai_engineering_role` and `llm_reason`.

In [ ]:
screened_jobs = pd.concat([jobs_df, results_df], axis=1)

## Save the Classified Jobs

The next notebooks read from `jobs/2-classified-jobs.jsonl`.

We save all rows, including rejected postings, because that makes the pipeline easier to inspect and debug.

In [ ]:
classified_jobs_path = Path("jobs") / "2-classified-jobs.jsonl"

screened_jobs.to_json(
    classified_jobs_path, orient="records", lines=True, force_ascii=False
)

print(f"Saved classified jobs to: {classified_jobs_path.resolve()}")

## Inspect the Results

Finally, display the title, classification, reason, and job URL.

Open a few accepted and rejected postings to check whether the model's reasoning matches the course definition of AI engineering.

In [ ]:
results_to_show = screened_jobs[
    ["title", "is_ai_engineering_role", "llm_reason", "job_url"]
].copy()
results_to_show["job_url"] = results_to_show["job_url"].apply(
    lambda url: f'<a href="{url}" target="_blank">{url}</a>' if pd.notna(url) else ""
)

display(HTML(results_to_show.to_html(escape=False, index=False)))